# AE 개선 비교

1. 기존 AE vs 개선 AE
2. 개선 AE vs 기존 PaDiM

앞부분은 기존 실습이며, 뒤의 **추가 비교 실험**에서 두 비교를 실행한다.

### 이상탐지를 위한 라이브러리 설치

In [ ]:
# !pip install anomalib
# !pip install pytorch-lightning

In [ ]:
# !pip install wget

## VisA 데이터셋 다운로드

In [ ]:
import os
import wget
import zipfile
from pathlib import Path
import pandas as pd
import tarfile

def download_visa_dataset():
    """VisA 데이터셋 다운로드"""
    base_url = "https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar"
    filename = "VisA_20220922.tar"
    extract_dir = "./VisA"


    if not os.path.exists(filename):
        print("VisA 데이터셋 다운로드 중...")
        wget.download(base_url, filename)
        print("\n다운로드 완료!")

    ## 압축 해제
    if not os.path.exists(extract_dir):
        os.makedirs(extract_dir, exist_ok=True)
        with tarfile.open(filename, 'r') as tar:
            tar.extractall(path=extract_dir)
        print("압축 해제 완료!")

def check_visa_structure(data_path="./VisA_pytorch/VisA"):
    """VisA 데이터셋 구조 확인"""
    data_path = Path(data_path)
    print(f"VisA 데이터셋 경로: {data_path}")

    categories = []
    for category_dir in data_path.iterdir():
        if category_dir.is_dir():
            categories.append(category_dir.name)
            print(f"\n카테고리: {category_dir.name}")

            # 이미지 개수 확인
            images_dir = category_dir / "Data" / "Images"
            if images_dir.exists():
                normal_count = len(list((images_dir / "Normal").glob("*"))) if (images_dir / "Normal").exists() else 0


                print(f"  정상 이미지: {normal_count}개")

    return categories

In [ ]:
download_visa_dataset()

In [ ]:
check_visa_structure(data_path="./VisA")

### Dataloader 구현

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import cv2
import numpy as np

class VisADataset(Dataset):
    def __init__(self, data_path, category, transform=None, is_train=True, include_mask=False, train_ratio=0.8):
        self.data_path = Path(data_path)
        self.category = category
        self.transform = transform
        self.is_train = is_train
        self.include_mask = include_mask

        self.image_paths = []
        self.mask_paths = []
        self.labels = []
        self.defect_types = []

        category_path = self.data_path / category

        # CSV 파일에서 메타데이터 로드
        csv_path = category_path / "image_anno.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            print(f"CSV 컬럼: {df.columns.tolist()}")

            # 정상/비정상 데이터 분리
            normal_df = df[df['label'] == 'normal'].copy()
            anomaly_df = df[df['label'] != 'normal'].copy()

            if self.is_train:
                # 훈련용: 정상 데이터의 일부만 사용
                train_size = int(len(normal_df) * train_ratio)
                selected_df = normal_df.iloc[:train_size]
                print(f"훈련용 정상 데이터: {len(selected_df)}개")
            else:
                # 테스트용: 나머지 정상 데이터 + 모든 비정상 데이터
                test_normal_size = len(normal_df) - int(len(normal_df) * train_ratio)
                test_normal_df = normal_df.iloc[-test_normal_size:] if test_normal_size > 0 else pd.DataFrame()
                selected_df = pd.concat([test_normal_df, anomaly_df], ignore_index=True)
                print(f"테스트용 정상 데이터: {len(test_normal_df)}개")
                print(f"테스트용 비정상 데이터: {len(anomaly_df)}개")

            # 데이터 로드
            for _, row in selected_df.iterrows():
                # 이미지 경로 처리 (중복 경로 제거)
                image_rel_path = row['image']
                if image_rel_path.startswith(category + '/'):
                    image_path = self.data_path / image_rel_path
                else:
                    image_path = category_path / image_rel_path

                if image_path.exists():
                    self.image_paths.append(str(image_path))
                    self.labels.append(0 if row['label'] == 'normal' else 1)
                    self.defect_types.append(row['label'])

                    # 마스크 경로 처리
                    if pd.notna(row['mask']) and self.include_mask:
                        mask_rel_path = row['mask']
                        if mask_rel_path.startswith(category + '/'):
                            mask_path = self.data_path / mask_rel_path
                        else:
                            mask_path = category_path / mask_rel_path
                        self.mask_paths.append(str(mask_path) if mask_path.exists() else None)
                    else:
                        self.mask_paths.append(None)
                else:
                    print(f"이미지 파일을 찾을 수 없습니다: {image_path}")
        else:
            print("CSV 파일이 없습니다. 폴더 구조를 기반으로 데이터를 로드합니다.")
            self._load_from_folder_structure()

    def _load_from_folder_structure(self):
        """폴더 구조를 기반으로 데이터 로드 (CSV가 없는 경우)"""
        category_path = self.data_path / self.category
        images_path = category_path / "Data" / "Images"
        masks_path = category_path / "Data" / "Masks"

        if self.is_train:
            # 훈련용: 정상 이미지만 사용
            normal_path = images_path / "Normal"
            if normal_path.exists():
                for img_path in normal_path.glob("*"):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        self.image_paths.append(str(img_path))
                        self.labels.append(0)  # 정상: 0
                        self.defect_types.append('normal')
                        self.mask_paths.append(None)
        else:
            # 테스트용: 정상 + 비정상 이미지
            # 정상 이미지
            normal_path = images_path / "Normal"
            if normal_path.exists():
                for img_path in normal_path.glob("*"):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        self.image_paths.append(str(img_path))
                        self.labels.append(0)
                        self.defect_types.append('normal')
                        self.mask_paths.append(None)

            # 비정상 이미지
            anomaly_path = images_path / "Anomaly"
            if anomaly_path.exists():
                for img_path in anomaly_path.glob("*"):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        self.image_paths.append(str(img_path))
                        self.labels.append(1)  # 이상: 1
                        self.defect_types.append('anomaly')

                        # 마스크 경로 찾기
                        if self.include_mask:
                            mask_name = img_path.stem + '.png'  # 확장자를 png로 변경
                            mask_path = masks_path / "Anomaly" / mask_name
                            self.mask_paths.append(str(mask_path) if mask_path.exists() else None)
                        else:
                            self.mask_paths.append(None)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 이미지 로드
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]

        # 마스크 로드 (있는 경우)
        mask = None
        if self.include_mask and self.mask_paths[idx] is not None:
            try:
                mask = Image.open(self.mask_paths[idx]).convert('L')
            except:
                mask = None

        # 변환 적용
        if self.transform:
            image = self.transform(image)
            if mask is not None:
                mask_transform = transforms.Compose([
                    transforms.Resize((256, 256)),
                    transforms.ToTensor()
                ])
                mask = mask_transform(mask)

        result = {
            'image': image,
            'label': label,
            'defect_type': self.defect_types[idx]
        }

        if mask is not None:
            result['mask'] = mask

        return result

In [ ]:
# 데이터 전처리 정의
def get_visa_transforms():
    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    test_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    return train_transform, test_transform


In [ ]:
def create_visa_data_loaders(data_path, category, batch_size=32, include_mask=False, train_ratio=0.8):
    """
    VisA 데이터셋용 데이터 로더 생성

    Args:
        data_path: VisA 데이터셋 경로
        category: 카테고리 이름
        batch_size: 배치 크기
        include_mask: 마스크 포함 여부
        train_ratio: 훈련 데이터 비율 (정상 데이터에서)
    """
    train_transform, test_transform = get_visa_transforms()

    train_dataset = VisADataset(
        data_path, category,
        transform=train_transform,
        is_train=True,
        include_mask=include_mask,
        train_ratio=train_ratio
    )

    test_dataset = VisADataset(
        data_path, category,
        transform=test_transform,
        is_train=False,
        include_mask=include_mask,
        train_ratio=train_ratio
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,  # Windows에서 문제가 있을 수 있으므로 낮춤
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    return train_loader, test_loader


## 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np

def visualize_visa_samples(dataset, num_samples=8):
    """VisA 데이터셋 샘플 시각화"""
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    indices = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)

    for i, idx in enumerate(indices):
        sample = dataset[idx]
        image = sample['image']
        label = sample['label']
        defect_type = sample['defect_type']

        # 텐서를 이미지로 변환
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0)
            image = image * torch.tensor([0.229, 0.224, 0.225]) + torch.tensor([0.485, 0.456, 0.406])
            image = torch.clamp(image, 0, 1)

        axes[i].imshow(image)
        title = f"{'Normal' if label == 0 else 'Anomaly'}"
        if defect_type != 'normal':
            title += f"\n({defect_type})"
        axes[i].set_title(title)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

def visualize_with_mask(dataset, idx):
    """이미지와 마스크 함께 시각화"""
    sample = dataset[idx]
    image = sample['image']
    mask = sample.get('mask')
    label = sample['label']
    defect_type = sample['defect_type']

    # 이미지 정규화 해제
    if isinstance(image, torch.Tensor):
        image = image.permute(1, 2, 0)
        image = image * torch.tensor([0.229, 0.224, 0.225]) + torch.tensor([0.485, 0.456, 0.406])
        image = torch.clamp(image, 0, 1)

    if mask is not None:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # 원본 이미지
        axes[0].imshow(image)
        axes[0].set_title('Original Image')
        axes[0].axis('off')

        # 마스크
        axes[1].imshow(mask.squeeze(), cmap='gray')
        axes[1].set_title('Anomaly Mask')
        axes[1].axis('off')

        # 오버레이
        overlay = image.clone()
        if mask.sum() > 0:
            mask_resized = torch.nn.functional.interpolate(
                mask.unsqueeze(0),
                size=image.shape[:2],
                mode='nearest'
            ).squeeze()
            overlay[mask_resized > 0.01] = torch.tensor([255.0, 0.0, 0.0])  # 빨간색으로 표시

        axes[2].imshow(overlay)
        axes[2].set_title(f'Overlay ({defect_type})')
        axes[2].axis('off')

    else:
        plt.figure(figsize=(8, 6))
        plt.imshow(image)
        plt.title(f"{'Normal' if label == 0 else 'Anomaly'} - {defect_type}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
def plot_visa_statistics(data_path, category):
    """VisA 데이터셋 통계 시각화"""
    category_path = Path(data_path) / category
    csv_path = category_path / "image_anno.csv"

    if csv_path.exists():
        df = pd.read_csv(csv_path)

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        title1 = 'Overall Class Distribution (Normal vs Anomaly)'
        title2 = 'Main Defect Types Distribution'
        title3 = 'Mask Availability Distribution'
        title4 = 'All Labels Distribution (Top 10)'
        xlabel2 = 'Number of Images'
        ylabel3 = 'Number of Images'
        xlabel4 = 'Number of Images'
        mask_labels = ['With Mask', 'Without Mask']

        # 전체 클래스 분포 (정상 vs 비정상)
        binary_labels = ['Normal' if label == 'normal' else 'Anomaly' for label in df['label']]
        class_counts = pd.Series(binary_labels).value_counts()
        axes[0, 0].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%')
        axes[0, 0].set_title(title1)

        # 결함 유형별 상세 분포
        anomaly_df = df[df['label'] != 'normal']
        if not anomaly_df.empty:
            # 결함 유형을 단순화 (콤마로 구분된 경우 첫 번째만 사용)
            simplified_defects = [label.split(',')[0] for label in anomaly_df['label']]
            defect_counts = pd.Series(simplified_defects).value_counts()

            axes[0, 1].barh(defect_counts.index, defect_counts.values)
            axes[0, 1].set_title(title2)
            axes[0, 1].set_xlabel(xlabel2)

        # 마스크 유무 분포
        mask_counts = df['mask'].notna().value_counts()
        mask_labels_mapped = [mask_labels[0] if has_mask else mask_labels[1] for has_mask in mask_counts.index]
        axes[1, 0].bar(mask_labels_mapped, mask_counts.values)
        axes[1, 0].set_title(title3)
        axes[1, 0].set_ylabel(ylabel3)

        # 전체 결함 유형 (복합 결함 포함)
        all_defect_counts = df['label'].value_counts()
        # 상위 10개만 표시
        top_defects = all_defect_counts.head(10)
        axes[1, 1].barh(range(len(top_defects)), top_defects.values)
        axes[1, 1].set_yticks(range(len(top_defects)))
        axes[1, 1].set_yticklabels(top_defects.index)
        axes[1, 1].set_title(title4)
        axes[1, 1].set_xlabel(xlabel4)

        plt.tight_layout()
        plt.show()

        # 데이터 요약 출력

        print(f"Total images: {len(df)}")
        print(f"Normal images: {len(df[df['label'] == 'normal'])}")
        print(f"Anomaly images: {len(df[df['label'] != 'normal'])}")
        print(f"Images with masks: {df['mask'].notna().sum()}")
        print(f"Unique defect types: {df['label'].nunique()}")

    else:
        error_msg = f"CSV 파일을 찾을 수 없습니다: {csv_path}" if KOREAN_FONT_AVAILABLE else f"CSV file not found: {csv_path}"
        print(error_msg)


In [ ]:
DATA_PATH = "./VisA" # <--- 실제 데이터셋 경로로 변경하세요!
CATEGORY_NAME = "pcb4" # VisA 데이터셋의 카테고리 중 하나 (예: 'pcb', 'capsules' 등)
BATCH_SIZE = 8 # 메모리에 따라 조절
D_REDUCED = 100 # PaDiM의 축소된 특징 차원
IMG_SIZE = 256  # 입력 이미지 크기 (예시)
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]
# PaDiM 내부에서 특징맵을 리사이즈할 크기 (고정)
PADIM_PATCH_SIZE = 28

train_loader, test_loader = create_visa_data_loaders(DATA_PATH, CATEGORY_NAME, BATCH_SIZE, include_mask=True)

In [ ]:
visualize_visa_samples(test_loader.dataset)
plot_visa_statistics(DATA_PATH, CATEGORY_NAME)

In [ ]:
visualize_with_mask(test_loader.dataset, 210)

## 모델 구현

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

import cv2
from PIL import Image
from pathlib import Path
import pickle
import json
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")


In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self, model_name='resnet18', layers_to_extract=['layer1', 'layer2', 'layer3']):
        super(FeatureExtractor, self).__init__()

        if model_name == 'resnet18':
            self.model = models.resnet18(pretrained=True)
        elif model_name == 'resnet50':
            self.model = models.resnet50(pretrained=True)
        elif model_name == 'wide_resnet50':
            self.model = models.wide_resnet50_2(pretrained=True)
        else:
            raise ValueError(f"지원하지 않는 모델: {model_name}")

        self.layers_to_extract = layers_to_extract
        self.features = {}

        # Hook 등록
        for name, module in self.model.named_modules():
            if name in layers_to_extract:
                module.register_forward_hook(self.save_features(name))

        # 평가 모드로 설정
        self.model.eval()

        # 그래디언트 계산 비활성화
        for param in self.model.parameters():
            param.requires_grad = False

    def save_features(self, name):
        def hook(module, input, output):
            self.features[name] = output
        return hook

    def forward(self, x):
        self.features.clear()
        _ = self.model(x)
        return self.features

# Feature Extractor 초기화
feature_extractor = FeatureExtractor('wide_resnet50', ['layer2', 'layer3']).to(device)

In [ ]:
class PaDiM:
    def __init__(self, feature_extractor, d_reduced=100):
        self.feature_extractor = feature_extractor
        self.d_reduced = d_reduced
        self.means = []
        self.inv_covariances = []
        self.projection_matrix = None

    def fit(self, train_loader):
        """정상 데이터로 모델 훈련"""
        print("PaDiM 훈련 시작...")

        # 모든 훈련 데이터에서 특징 추출
        all_features = []

        for batch in tqdm(train_loader, desc="특징 추출 중"):
            images = batch['image'].to(device)

            with torch.no_grad():
                features = self.feature_extractor(images)

                # 각 이미지별로 처리
                for i in range(images.size(0)):
                    img_features = []

                    # 레이어별 특징 추출 및 결합
                    for layer_name, feat in features.items():
                        # 단일 이미지 특징 추출
                        single_feat = feat[i:i+1]  # [1, C, H, W]

                        # 28x28로 리사이즈
                        feat_resized = F.adaptive_avg_pool2d(single_feat, (28, 28))

                        # 패치별 특징으로 변환 [784, C]
                        feat_patches = feat_resized.permute(0, 2, 3, 1).reshape(28*28, -1)
                        img_features.append(feat_patches)

                    # 레이어별 특징 결합 [784, total_channels]
                    combined_features = torch.cat(img_features, dim=1)
                    all_features.append(combined_features)

        # 모든 특징을 하나로 결합 [N*784, total_channels]
        all_features = torch.cat(all_features, dim=0)
        print(f"전체 특징 크기: {all_features.shape}")

        # 차원 축소를 위한 Random Projection 행렬 생성
        if all_features.shape[1] > self.d_reduced:
            self.projection_matrix = torch.randn(all_features.shape[1], self.d_reduced).to(device)
            # 정규화
            self.projection_matrix = F.normalize(self.projection_matrix, dim=0)

            # 차원 축소 적용
            all_features = torch.mm(all_features, self.projection_matrix)
            print(f"차원 축소 후 크기: {all_features.shape}")

        # 패치 위치별로 분리하여 통계 계산
        n_samples = len(all_features) // (28 * 28)

        for patch_idx in range(28 * 28):
            # 특정 패치 위치의 모든 특징 추출
            patch_features = all_features[patch_idx::28*28]  # [n_samples, d_reduced]

            # 평균과 공분산 계산
            mean = torch.mean(patch_features, dim=0)

            # 공분산 행렬 계산 (정규화 추가)
            centered_features = patch_features - mean
            cov = torch.mm(centered_features.T, centered_features) / (patch_features.shape[0] - 1)

            # 수치 안정성을 위한 정규화
            cov += torch.eye(cov.shape[0]).to(device) * 1e-4

            # 역행렬 계산
            try:
                inv_cov = torch.linalg.inv(cov)
            except:
                # 특이행렬인 경우 pseudo-inverse 사용
                inv_cov = torch.linalg.pinv(cov)

            self.means.append(mean)
            self.inv_covariances.append(inv_cov)

        print(f"PaDiM 훈련 완료! (패치 수: {len(self.means)})")

    def predict(self, test_loader):
        """테스트 데이터에 대한 이상 점수 계산"""
        anomaly_scores = []

        for batch in tqdm(test_loader, desc="이상 탐지 중"):
            images = batch['image'].to(device)

            with torch.no_grad():
                features = self.feature_extractor(images)

                # 각 이미지별로 처리
                for i in range(images.size(0)):
                    img_features = []

                    # 레이어별 특징 추출 및 결합
                    for layer_name, feat in features.items():
                        single_feat = feat[i:i+1]
                        feat_resized = F.adaptive_avg_pool2d(single_feat, (28, 28))
                        feat_patches = feat_resized.permute(0, 2, 3, 1).reshape(28*28, -1)
                        img_features.append(feat_patches)

                    combined_features = torch.cat(img_features, dim=1)

                    # 차원 축소 (훈련 시와 동일한 방식)
                    if self.projection_matrix is not None:
                        combined_features = torch.mm(combined_features, self.projection_matrix)

                    # 각 패치별 Mahalanobis 거리 계산
                    patch_scores = []
                    for patch_idx in range(28 * 28):
                        diff = combined_features[patch_idx] - self.means[patch_idx]

                        # Mahalanobis 거리 계산
                        mahal_dist = torch.sqrt(torch.mm(
                            torch.mm(diff.unsqueeze(0), self.inv_covariances[patch_idx]),
                            diff.unsqueeze(1)
                        )).item()

                        patch_scores.append(mahal_dist)

                    # 전체 이상 점수 (최대값 사용)
                    anomaly_scores.append(max(patch_scores))

        return np.array(anomaly_scores), None

In [ ]:
class ConvAutoEncoder(nn.Module):
    def __init__(self, input_channels=3, latent_dim=128):
        super(ConvAutoEncoder, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(input_channels, 32, 4, stride=2, padding=1),  # 128x128
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # 64x64
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),  # 32x32
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),  # 16x16
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, latent_dim, 4, stride=2, padding=1),  # 8x8
            nn.BatchNorm2d(latent_dim),
            nn.ReLU()
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, 4, stride=2, padding=1),  # 16x16
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # 32x32
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # 64x64
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),  # 128x128
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, input_channels, 4, stride=2, padding=1),  # 256x256
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

class AutoEncoderAnomalyDetector:
    def __init__(self, input_size=(256, 256), latent_dim=128):
        self.model = ConvAutoEncoder(input_channels=3, latent_dim=latent_dim).to(device)
        self.criterion = nn.MSELoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001)
        self.input_size = input_size

    def fit(self, train_loader, epochs=50):
        """AutoEncoder 훈련"""
        print("AutoEncoder 훈련 시작...")
        self.model.train()

        train_losses = []

        for epoch in range(epochs):
            epoch_loss = 0
            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
                images = batch['image'].to(device)

                self.optimizer.zero_grad()
                reconstructed = self.model(images)
                loss = self.criterion(reconstructed, images)
                loss.backward()
                self.optimizer.step()

                epoch_loss += loss.item()

            avg_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_loss)
            print(f"Epoch {epoch+1}, Loss: {avg_loss:.6f}")

        self.model.eval()
        print("AutoEncoder 훈련 완료!")
        return train_losses

    def predict(self, test_loader):
        """테스트 데이터에 대한 재구성 오차 계산"""
        self.model.eval()
        anomaly_scores = []

        with torch.no_grad():
            for batch in tqdm(test_loader, desc="이상 탐지 중"):
                images = batch['image'].to(device)
                reconstructed = self.model(images)

                # 픽셀별 재구성 오차
                reconstruction_errors = F.mse_loss(reconstructed, images, reduction='none')
                reconstruction_errors = reconstruction_errors.view(images.size(0), -1).mean(dim=1)

                anomaly_scores.extend(reconstruction_errors.cpu().numpy())

        return np.array(anomaly_scores)

In [ ]:
class OneClassSVMAnomalyDetector:
    def __init__(self, feature_extractor, nu=0.1):
        self.feature_extractor = feature_extractor
        self.svm = OneClassSVM(nu=nu, kernel='rbf', gamma='scale')
        self.scaler = StandardScaler()

    def extract_features(self, data_loader):
        """특징 추출"""
        features = []

        for batch in tqdm(data_loader, desc="특징 추출 중"):
            images = batch['image'].to(device)

            with torch.no_grad():
                feat_dict = self.feature_extractor(images)

                # 모든 레이어의 특징을 결합하고 평균 풀링
                combined_features = []
                for layer_name, feat in feat_dict.items():
                    pooled_feat = F.adaptive_avg_pool2d(feat, (1, 1))
                    combined_features.append(pooled_feat.view(feat.size(0), -1))

                batch_features = torch.cat(combined_features, dim=1)
                features.append(batch_features.cpu().numpy())

        return np.concatenate(features, axis=0)

    def fit(self, train_loader):
        """One-Class SVM 훈련"""
        print("One-Class SVM 훈련 시작...")

        # 특징 추출
        train_features = self.extract_features(train_loader)

        # 정규화
        train_features_scaled = self.scaler.fit_transform(train_features)

        # SVM 훈련
        self.svm.fit(train_features_scaled)
        print("One-Class SVM 훈련 완료!")

    def predict(self, test_loader):
        """테스트 데이터에 대한 이상 점수 계산"""
        # 특징 추출
        test_features = self.extract_features(test_loader)

        # 정규화
        test_features_scaled = self.scaler.transform(test_features)

        # 이상 점수 계산 (결정 함수의 음수)
        anomaly_scores = -self.svm.decision_function(test_features_scaled)

        return anomaly_scores

In [ ]:
class AnomalyDetectionEvaluator:
    def __init__(self):
        self.results = {}

    def evaluate(self, y_true, y_scores, method_name):
        """평가 메트릭 계산"""
        # AUROC
        auroc = roc_auc_score(y_true, y_scores)

        # AUPRC
        precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
        auprc = auc(recall, precision)

        # 최적 임계값에서의 F1 스코어
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

        # NaN 값 제거
        valid_indices = ~np.isnan(f1_scores)
        if np.any(valid_indices):
            valid_f1_scores = f1_scores[valid_indices]
            valid_thresholds = thresholds[valid_indices] if len(thresholds) == len(f1_scores) else thresholds[:len(valid_f1_scores)]

            best_f1_idx = np.argmax(valid_f1_scores)
            best_f1 = valid_f1_scores[best_f1_idx]
            best_threshold = valid_thresholds[best_f1_idx] if len(valid_thresholds) > best_f1_idx else 0.5
        else:
            best_f1 = 0.0
            best_threshold = 0.5

        self.results[method_name] = {
            'AUROC': auroc,
            'AUPRC': auprc,
            'Best_F1': best_f1,
            'Best_Threshold': best_threshold,
            'y_true': y_true,
            'y_scores': y_scores
        }

        print(f"{method_name} 결과:")
        print(f"  AUROC: {auroc:.4f}")
        print(f"  AUPRC: {auprc:.4f}")
        print(f"  Best F1: {best_f1:.4f}")
        print()

        return auroc, auprc, best_f1

    def plot_roc_curve(self):
        """ROC 곡선 그리기"""
        plt.figure(figsize=(10, 8))

        for method_name, result in self.results.items():
            fpr, tpr, _ = roc_curve(result['y_true'], result['y_scores'])
            plt.plot(fpr, tpr, label=f"{method_name} (AUC = {result['AUROC']:.3f})")

        plt.plot([0, 1], [0, 1], 'k--', label='Random')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curves Comparison')
        plt.legend()
        plt.grid(True)
        plt.show()

    def plot_precision_recall_curve(self):
        """PR 곡선 그리기"""
        plt.figure(figsize=(10, 8))

        for method_name, result in self.results.items():
            precision, recall, _ = precision_recall_curve(result['y_true'], result['y_scores'])
            plt.plot(recall, precision, label=f"{method_name} (AUC = {result['AUPRC']:.3f})")

        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title('Precision-Recall Curves Comparison')
        plt.legend()
        plt.grid(True)
        plt.show()

    def plot_score_distribution(self):
        """이상 점수 분포 시각화"""
        fig, axes = plt.subplots(len(self.results), 1, figsize=(12, 4*len(self.results)))
        if len(self.results) == 1:
            axes = [axes]

        for i, (method_name, result) in enumerate(self.results.items()):
            normal_scores = result['y_scores'][result['y_true'] == 0]
            anomaly_scores = result['y_scores'][result['y_true'] == 1]

            axes[i].hist(normal_scores, bins=50, alpha=0.7, label='Normal', density=True)
            axes[i].hist(anomaly_scores, bins=50, alpha=0.7, label='Anomaly', density=True)
            axes[i].axvline(result['Best_Threshold'], color='red', linestyle='--',
                           label=f"Best Threshold: {result['Best_Threshold']:.3f}")
            axes[i].set_title(f'{method_name} - Anomaly Score Distribution')
            axes[i].set_xlabel('Anomaly Score')
            axes[i].set_ylabel('Density')
            axes[i].legend()
            axes[i].grid(True)

        plt.tight_layout()
        plt.show()

    def compare_methods(self):
        """방법별 성능 비교"""
        methods = list(self.results.keys())
        aurocs = [self.results[m]['AUROC'] for m in methods]
        auprcs = [self.results[m]['AUPRC'] for m in methods]
        f1_scores = [self.results[m]['Best_F1'] for m in methods]

        x = np.arange(len(methods))
        width = 0.25

        plt.figure(figsize=(12, 6))
        plt.bar(x - width, aurocs, width, label='AUROC', alpha=0.8)
        plt.bar(x, auprcs, width, label='AUPRC', alpha=0.8)
        plt.bar(x + width, f1_scores, width, label='Best F1', alpha=0.8)

        plt.xlabel('Methods')
        plt.ylabel('Score')
        plt.title('Performance Comparison')
        plt.xticks(x, methods, rotation=45)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

In [ ]:
class VisAAnomalyDetectionSystem:
    def __init__(self, data_path, category):
        self.data_path = data_path
        self.category = category
        self.evaluator = AnomalyDetectionEvaluator()

        # 데이터 로더 생성
        self.train_loader, self.test_loader = create_visa_data_loaders(
            data_path, category, batch_size=16, include_mask=False, train_ratio=0.8
        )

        # 테스트 레이블 추출
        self.test_labels = []
        for batch in self.test_loader:
            self.test_labels.extend(batch['label'].numpy())
        self.test_labels = np.array(self.test_labels)

        print(f"시스템 초기화 완료:")
        print(f"  카테고리: {category}")
        print(f"  훈련 데이터: {len(self.train_loader.dataset)}개")
        print(f"  테스트 데이터: {len(self.test_loader.dataset)}개")
        print(f"  테스트 중 이상 데이터: {self.test_labels.sum()}개")

    def run_padim(self):
        """PaDiM 실행"""
        print("\n=== PaDiM 실행 ===")
        padim = PaDiM(feature_extractor, d_reduced=100)
        padim.fit(self.train_loader)
        scores, _ = padim.predict(self.test_loader)
        self.evaluator.evaluate(self.test_labels, scores, 'PaDiM')
        return scores

    def run_autoencoder(self, epochs=30):
        """AutoEncoder 실행"""
        print("\n=== AutoEncoder 실행 ===")
        ae_detector = AutoEncoderAnomalyDetector()
        ae_detector.fit(self.train_loader, epochs=epochs)
        scores = ae_detector.predict(self.test_loader)
        self.evaluator.evaluate(self.test_labels, scores, 'AutoEncoder')
        return scores

    def run_ocsvm(self):
        """One-Class SVM 실행"""
        print("\n=== One-Class SVM 실행 ===")
        ocsvm_detector = OneClassSVMAnomalyDetector(feature_extractor, nu=0.1)
        ocsvm_detector.fit(self.train_loader)
        scores = ocsvm_detector.predict(self.test_loader)
        self.evaluator.evaluate(self.test_labels, scores, 'One-Class SVM')
        return scores

    def run_all_methods(self):
        """모든 방법 실행 및 비교"""
        print("모든 anomaly detection 방법을 실행합니다...")

        # 각 방법 실행 (오류가 발생해도 계속 진행)
        methods_tried = []

        # 1. AutoEncoder (가장 안정적)
        try:
            print("\n=== AutoEncoder 실행 ===")
            self.run_autoencoder(epochs=15)
            methods_tried.append("AutoEncoder")
        except Exception as e:
            print(f"AutoEncoder 실행 중 오류: {e}")

        # 2. One-Class SVM
        try:
            print("\n=== One-Class SVM 실행 ===")
            self.run_ocsvm()
            methods_tried.append("One-Class SVM")
        except Exception as e:
            print(f"One-Class SVM 실행 중 오류: {e}")

        # 3. PaDiM (메모리 사용량이 많을 수 있음)
        try:
            print("\n=== PaDiM 실행 ===")
            self.run_padim()
            methods_tried.append("PaDiM")
        except Exception as e:
            print(f"PaDiM 실행 중 오류: {e}")
            print("팁: GPU 메모리가 부족할 수 있습니다. 배치 크기를 줄여보세요.")

        # 결과 시각화
        if self.evaluator.results:
            print(f"\n=== 전체 결과 ({len(methods_tried)}개 방법 성공) ===")
            try:
                self.evaluator.compare_methods()
            except Exception as e:
                print(f"비교 차트 생성 중 오류: {e}")

            try:
                self.evaluator.plot_roc_curve()
            except Exception as e:
                print(f"ROC 곡선 생성 중 오류: {e}")

            try:
                self.evaluator.plot_precision_recall_curve()
            except Exception as e:
                print(f"PR 곡선 생성 중 오류: {e}")

            try:
                self.evaluator.plot_score_distribution()
            except Exception as e:
                print(f"점수 분포 차트 생성 중 오류: {e}")
        else:
            print("❌ 성공적으로 실행된 방법이 없습니다.")
            print("다음을 확인해보세요:")
            print("1. 데이터 경로가 올바른지 확인")
            print("2. GPU 메모리가 충분한지 확인")
            print("3. 필요한 라이브러리가 모두 설치되어 있는지 확인")

    def save_results(self, filename):
        """결과 저장"""
        results_to_save = {}
        for method, result in self.evaluator.results.items():
            results_to_save[method] = {
                'AUROC': result['AUROC'],
                'AUPRC': result['AUPRC'],
                'Best_F1': result['Best_F1'],
                'Best_Threshold': result['Best_Threshold']
            }

        with open(filename, 'w') as f:
            json.dump(results_to_save, f, indent=2)
        print(f"결과가 {filename}에 저장되었습니다.")

In [ ]:
system = VisAAnomalyDetectionSystem("./VisA", "pcb4")

# 개별 방법 실행
padim_scores = system.run_padim()
ae_scores = system.run_autoencoder(epochs=50)
svm_scores = system.run_ocsvm()


In [ ]:
system.evaluator.compare_methods()

In [ ]:
system.evaluator.plot_roc_curve()

In [ ]:
system.evaluator.plot_precision_recall_curve()

In [ ]:
system.evaluator.plot_score_distribution()

### 실시간 inference

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2
import numpy as np
import torchvision.models as models

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

import cv2
from PIL import Image
from pathlib import Path
import pickle
import json
from tqdm import tqdm
import warnings

In [ ]:
class VisADataset(Dataset):
    def __init__(self, data_path, category, transform=None, is_train=True, include_mask=False, train_ratio=0.8):
        self.data_path = Path(data_path)
        self.category = category
        self.transform = transform
        self.is_train = is_train
        self.include_mask = include_mask

        self.image_paths = []
        self.mask_paths = []
        self.labels = []
        self.defect_types = []

        category_path = self.data_path / category

        # CSV 파일에서 메타데이터 로드
        csv_path = category_path / "image_anno.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            print(f"CSV 컬럼: {df.columns.tolist()}")

            # 정상/비정상 데이터 분리
            normal_df = df[df['label'] == 'normal'].copy()
            anomaly_df = df[df['label'] != 'normal'].copy()

            if self.is_train:
                # 훈련용: 정상 데이터의 일부만 사용
                train_size = int(len(normal_df) * train_ratio)
                selected_df = normal_df.iloc[:train_size]
                print(f"훈련용 정상 데이터: {len(selected_df)}개")
            else:
                # 테스트용: 나머지 정상 데이터 + 모든 비정상 데이터
                test_normal_size = len(normal_df) - int(len(normal_df) * train_ratio)
                test_normal_df = normal_df.iloc[-test_normal_size:] if test_normal_size > 0 else pd.DataFrame()
                selected_df = pd.concat([test_normal_df, anomaly_df], ignore_index=True)
                print(f"테스트용 정상 데이터: {len(test_normal_df)}개")
                print(f"테스트용 비정상 데이터: {len(anomaly_df)}개")

            # 데이터 로드
            for _, row in selected_df.iterrows():
                # 이미지 경로 처리 (중복 경로 제거)
                image_rel_path = row['image']
                if image_rel_path.startswith(category + '/'):
                    image_path = self.data_path / image_rel_path
                else:
                    image_path = category_path / image_rel_path

                if image_path.exists():
                    self.image_paths.append(str(image_path))
                    self.labels.append(0 if row['label'] == 'normal' else 1)
                    self.defect_types.append(row['label'])

                    # 마스크 경로 처리
                    if pd.notna(row['mask']) and self.include_mask:
                        mask_rel_path = row['mask']
                        if mask_rel_path.startswith(category + '/'):
                            mask_path = self.data_path / mask_rel_path
                        else:
                            mask_path = category_path / mask_rel_path
                        self.mask_paths.append(str(mask_path) if mask_path.exists() else None)
                    else:
                        self.mask_paths.append(None)
                else:
                    print(f"이미지 파일을 찾을 수 없습니다: {image_path}")
        else:
            print("CSV 파일이 없습니다. 폴더 구조를 기반으로 데이터를 로드합니다.")
            self._load_from_folder_structure()

    def _load_from_folder_structure(self):
        """폴더 구조를 기반으로 데이터 로드 (CSV가 없는 경우)"""
        category_path = self.data_path / self.category
        images_path = category_path / "Data" / "Images"
        masks_path = category_path / "Data" / "Masks"

        if self.is_train:
            # 훈련용: 정상 이미지만 사용
            normal_path = images_path / "Normal"
            if normal_path.exists():
                for img_path in normal_path.glob("*"):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        self.image_paths.append(str(img_path))
                        self.labels.append(0)  # 정상: 0
                        self.defect_types.append('normal')
                        self.mask_paths.append(None)
        else:
            # 테스트용: 정상 + 비정상 이미지
            # 정상 이미지
            normal_path = images_path / "Normal"
            if normal_path.exists():
                for img_path in normal_path.glob("*"):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        self.image_paths.append(str(img_path))
                        self.labels.append(0)
                        self.defect_types.append('normal')
                        self.mask_paths.append(None)

            # 비정상 이미지
            anomaly_path = images_path / "Anomaly"
            if anomaly_path.exists():
                for img_path in anomaly_path.glob("*"):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        self.image_paths.append(str(img_path))
                        self.labels.append(1)  # 이상: 1
                        self.defect_types.append('anomaly')

                        # 마스크 경로 찾기
                        if self.include_mask:
                            mask_name = img_path.stem + '.png'  # 확장자를 png로 변경
                            mask_path = masks_path / "Anomaly" / mask_name
                            self.mask_paths.append(str(mask_path) if mask_path.exists() else None)
                        else:
                            self.mask_paths.append(None)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 이미지 로드
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]

        # 마스크 로드 (있는 경우)
        mask = None
        if self.include_mask and self.mask_paths[idx] is not None:
            try:
                mask = Image.open(self.mask_paths[idx]).convert('L')
            except:
                mask = None

        # 변환 적용
        if self.transform:
            image = self.transform(image)
            if mask is not None:
                mask_transform = transforms.Compose([
                    transforms.Resize((256, 256)),
                    transforms.ToTensor()
                ])
                mask = mask_transform(mask)

        result = {
            'image': image,
            'label': label,
            'defect_type': self.defect_types[idx]
        }

        if mask is not None:
            result['mask'] = mask

        return result

In [ ]:
def create_visa_data_loaders(data_path, category, batch_size=32, include_mask=False, train_ratio=0.8):
    """
    VisA 데이터셋용 데이터 로더 생성

    Args:
        data_path: VisA 데이터셋 경로
        category: 카테고리 이름
        batch_size: 배치 크기
        include_mask: 마스크 포함 여부
        train_ratio: 훈련 데이터 비율 (정상 데이터에서)
    """
    train_transform, test_transform = get_visa_transforms()

    train_dataset = VisADataset(
        data_path, category,
        transform=train_transform,
        is_train=True,
        include_mask=include_mask,
        train_ratio=train_ratio
    )

    test_dataset = VisADataset(
        data_path, category,
        transform=test_transform,
        is_train=False,
        include_mask=include_mask,
        train_ratio=train_ratio
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,  # Windows에서 문제가 있을 수 있으므로 낮춤
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    return train_loader, test_loader


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self, model_name='resnet18', layers_to_extract=['layer1', 'layer2', 'layer3']):
        super(FeatureExtractor, self).__init__()

        if model_name == 'resnet18':
            self.model = models.resnet18(pretrained=True)
        elif model_name == 'resnet50':
            self.model = models.resnet50(pretrained=True)
        elif model_name == 'wide_resnet50':
            self.model = models.wide_resnet50_2(pretrained=True)
        else:
            raise ValueError(f"지원하지 않는 모델: {model_name}")

        self.layers_to_extract = layers_to_extract
        self.features = {}

        # Hook 등록
        for name, module in self.model.named_modules():
            if name in layers_to_extract:
                module.register_forward_hook(self.save_features(name))

        # 평가 모드로 설정
        self.model.eval()

        # 그래디언트 계산 비활성화
        for param in self.model.parameters():
            param.requires_grad = False

    def save_features(self, name):
        def hook(module, input, output):
            self.features[name] = output
        return hook

    def forward(self, x):
        self.features.clear()
        _ = self.model(x)
        return self.features

# Feature Extractor 초기화
feature_extractor = FeatureExtractor('wide_resnet50', ['layer2', 'layer3']).to(device)


In [ ]:
# --- PaDiM 클래스 (사용자 제공 코드, predict 수정) ---
class PaDiM:
    def __init__(self, feature_extractor, d_reduced=100, padim_patch_size=PADIM_PATCH_SIZE):
        self.feature_extractor = feature_extractor
        self.d_reduced = d_reduced
        self.padim_patch_size = padim_patch_size
        self.means = []
        self.inv_covariances = []
        self.projection_matrix = None

    def fit(self, train_loader):
        print("PaDiM 훈련 시작...")
        all_features = []
        for batch in tqdm(train_loader, desc="특징 추출 중"):
            images = batch['image'].to(device)
            with torch.no_grad():
                features = self.feature_extractor(images)
                for i in range(images.size(0)):
                    img_features = []
                    for layer_name, feat in features.items():
                        single_feat = feat[i:i+1]
                        feat_resized = F.adaptive_avg_pool2d(single_feat, (self.padim_patch_size, self.padim_patch_size))
                        feat_patches = feat_resized.permute(0, 2, 3, 1).reshape(self.padim_patch_size*self.padim_patch_size, -1)
                        img_features.append(feat_patches)
                    combined_features = torch.cat(img_features, dim=1)
                    all_features.append(combined_features)

        all_features = torch.cat(all_features, dim=0)
        print(f"전체 특징 크기: {all_features.shape}")

        if all_features.shape[1] > self.d_reduced:
            self.projection_matrix = torch.randn(all_features.shape[1], self.d_reduced).to(device)
            self.projection_matrix = F.normalize(self.projection_matrix, dim=0)
            all_features = torch.mm(all_features, self.projection_matrix)
            print(f"차원 축소 후 크기: {all_features.shape}")

        num_patches = self.padim_patch_size * self.padim_patch_size
        for patch_idx in range(num_patches):
            patch_features = all_features[patch_idx::num_patches]
            mean = torch.mean(patch_features, dim=0)
            centered_features = patch_features - mean
            cov = torch.mm(centered_features.T, centered_features) / (patch_features.shape[0] - 1)
            cov += torch.eye(cov.shape[0]).to(device) * 1e-4
            try:
                inv_cov = torch.linalg.inv(cov)
            except RuntimeError:
                inv_cov = torch.linalg.pinv(cov)
            self.means.append(mean)
            self.inv_covariances.append(inv_cov)
        print(f"PaDiM 훈련 완료! (패치 수: {len(self.means)})")

    def predict(self, images_tensor): # DataLoader 대신 이미지 텐서를 직접 받도록 수정
        """테스트 이미지 텐서에 대한 이상 점수 및 이상 맵 계산"""
        all_image_scores = []
        all_anomaly_maps = []

        with torch.no_grad():
            features = self.feature_extractor(images_tensor) # 이미지 텐서 바로 사용

            for i in range(images_tensor.size(0)):
                img_features = []
                for layer_name, feat in features.items():
                    single_feat = feat[i:i+1]
                    feat_resized = F.adaptive_avg_pool2d(single_feat, (self.padim_patch_size, self.padim_patch_size))
                    feat_patches = feat_resized.permute(0, 2, 3, 1).reshape(self.padim_patch_size*self.padim_patch_size, -1)
                    img_features.append(feat_patches)

                combined_features = torch.cat(img_features, dim=1)

                if self.projection_matrix is not None:
                    combined_features = torch.mm(combined_features, self.projection_matrix)

                patch_scores_for_image = []
                num_patches = self.padim_patch_size * self.padim_patch_size
                for patch_idx in range(num_patches):
                    diff = combined_features[patch_idx] - self.means[patch_idx]
                    mahal_dist = torch.sqrt(torch.mm(
                        torch.mm(diff.unsqueeze(0), self.inv_covariances[patch_idx]),
                        diff.unsqueeze(1)
                    )).item()
                    patch_scores_for_image.append(mahal_dist)

                all_image_scores.append(max(patch_scores_for_image))

                # 이상 맵 생성 (28x28 형태로)
                anomaly_map = torch.tensor(patch_scores_for_image).reshape(self.padim_patch_size, self.padim_patch_size).cpu().numpy()
                all_anomaly_maps.append(anomaly_map)

        return np.array(all_image_scores), np.array(all_anomaly_maps)

In [ ]:
def get_visa_transforms():
    """
    VisA 데이터셋에 적용할 이미지 변환(transform)을 반환합니다.
    실제 데이터셋과 모델에 맞게 이 부분을 수정해야 합니다.
    """
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD)
    ])

    # 테스트 시에는 일반적으로 Augmentation을 많이 하지 않습니다.
    test_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD)
    ])
    return train_transform, test_transform


In [ ]:
# --- 데이터셋 경로 및 카테고리 설정 ---
# !!! 중요: 이 경로를 실제 VisA 데이터셋이 있는 경로로 수정해야 합니다 !!!
# 예시: DATA_PATH = "/path/to/your/visa_dataset"
# VisA 데이터셋이 없다면, 아래 예시처럼 직접 폴더를 만들고 소수의 이미지를 넣어 테스트할 수 있습니다.
# DATA_PATH/CATEGORY_NAME/train/good/img1.png, img2.png ...
# DATA_PATH/CATEGORY_NAME/test/good/img1.png, img2.png ...
# DATA_PATH/CATEGORY_NAME/test/anomaly_type1/img1.png, img2.png ...
DATA_PATH = "./VisA" # <--- 실제 데이터셋 경로로 변경하세요!
CATEGORY_NAME = "pcb4" # VisA 데이터셋의 카테고리 중 하나 (예: 'pcb', 'capsules' 등)
BATCH_SIZE = 8 # 메모리에 따라 조절
D_REDUCED = 100 # PaDiM의 축소된 특징 차원
IMG_SIZE = 256  # 입력 이미지 크기 (예시)
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]
# PaDiM 내부에서 특징맵을 리사이즈할 크기 (고정)
PADIM_PATCH_SIZE = 28

# 결과물 저장 경로
OUTPUT_DIR = "padim_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
SAVED_MODEL_PATH = os.path.join(OUTPUT_DIR, "padim_model_state.pkl")

print(f"Data path set to: {DATA_PATH}")
print(f"Make sure you have data in the respective folders, e.g., {os.path.join(DATA_PATH, CATEGORY_NAME, 'train', 'good')}")
print("-" * 30)


# --- 모델 초기화 ---
print("FeatureExtractor 초기화 중...")
# 사용할 CNN 백본과 추출할 레이어 설정
# 예: wide_resnet50_2의 layer1, layer2, layer3 특징 사용
feature_extractor = FeatureExtractor(model_name='wide_resnet50', layers_to_extract=['layer1', 'layer2', 'layer3']).to(device)
print("PaDiM 모델 초기화 중...")
padim_model = PaDiM(feature_extractor=feature_extractor, d_reduced=D_REDUCED)

# --- 데이터 로더 생성 ---
print("데이터 로더 생성 중...")
try:
    train_loader, test_loader = create_visa_data_loaders(DATA_PATH, CATEGORY_NAME, BATCH_SIZE)

    # 훈련 데이터셋에 이미지가 있는지 확인
    if len(train_loader.dataset) == 0:
        print(f"훈련 데이터셋에 이미지가 없습니다! 경로를 확인하세요: {os.path.join(DATA_PATH, CATEGORY_NAME, 'train', 'good')}")
        # 이 경우 학습을 진행할 수 없으므로, 예외를 발생시키거나 중단할 수 있습니다.
        # raise ValueError("훈련 데이터가 비어있습니다. 학습을 진행할 수 없습니다.")
    else:
        print(f"훈련 데이터셋 크기: {len(train_loader.dataset)}")
        print(f"테스트 데이터셋 크기: {len(test_loader.dataset)}")

        # --- PaDiM 모델 학습 ---
        padim_model.fit(train_loader)

        # --- 학습된 모델 저장 ---
        print(f"학습된 PaDiM 모델을 '{SAVED_MODEL_PATH}'에 저장합니다...")
        # PaDiM 클래스 전체를 저장하거나, 필요한 attribute만 저장할 수 있습니다.
        # 여기서는 주요 attribute들을 딕셔너리 형태로 저장합니다.
        padim_state = {
            'means': padim_model.means,
            'inv_covariances': padim_model.inv_covariances,
            'projection_matrix': padim_model.projection_matrix,
            'd_reduced': padim_model.d_reduced,
            'padim_patch_size': padim_model.padim_patch_size
            # FeatureExtractor는 구조가 동일하므로 저장하지 않고 추론 시 새로 생성
        }
        with open(SAVED_MODEL_PATH, 'wb') as f:
            pickle.dump(padim_state, f)
        print("모델 저장 완료.")

except FileNotFoundError as e:
    print(f"데이터 경로 오류: {e}")
    print("DATA_PATH 및 CATEGORY_NAME을 올바르게 설정하고, 해당 경로에 이미지 파일이 있는지 확인해주세요.")
except Exception as e:
    print(f"학습 중 오류 발생: {e}")



In [ ]:
# --- 저장된 PaDiM 모델 로드 함수 ---
def load_padim_model_state(filepath, feature_extractor_instance):
    if not os.path.exists(filepath):
        print(f"저장된 모델 파일이 없습니다: {filepath}")
        return None

    with open(filepath, 'rb') as f:
        padim_state = pickle.load(f)

    # 새 PaDiM 인스턴스를 만들고 상태를 로드
    loaded_model = PaDiM(feature_extractor_instance,
                           d_reduced=padim_state.get('d_reduced', 100), # 기본값 설정
                           padim_patch_size=padim_state.get('padim_patch_size', PADIM_PATCH_SIZE)) # 기본값 설정
    loaded_model.means = padim_state['means']
    loaded_model.inv_covariances = padim_state['inv_covariances']
    loaded_model.projection_matrix = padim_state['projection_matrix']

    print(f"'{filepath}'에서 PaDiM 모델 상태 로드 완료.")
    return loaded_model

# --- Gradio 추론 함수 ---
# 모델 로드는 Gradio 앱 시작 시 한 번만 수행하도록 전역 변수로 관리
gradio_feature_extractor = None
gradio_padim_model = None
_, gradio_test_transform = get_visa_transforms() # 테스트용 변환 사용

def initialize_gradio_models():
    global gradio_feature_extractor, gradio_padim_model
    if gradio_feature_extractor is None:
        print("Gradio: FeatureExtractor 초기화 중...")
        gradio_feature_extractor = FeatureExtractor(model_name='wide_resnet50', layers_to_extract=['layer1', 'layer2', 'layer3']).to(device)

    if gradio_padim_model is None:
        print("Gradio: 저장된 PaDiM 모델 로드 중...")
        gradio_padim_model = load_padim_model_state(SAVED_MODEL_PATH, gradio_feature_extractor)
        if gradio_padim_model is None:
            print("Gradio: 모델 로드 실패! 학습을 먼저 실행하거나 모델 경로를 확인하세요.")
            # Gradio 인터페이스 실행 전에 모델이 로드되었는지 확인 필요

# Gradio 앱 시작 전에 모델 초기화 시도
initialize_gradio_models()


def padim_inference_gradio(input_image_pil):
    """Gradio 인터페이스를 위한 추론 함수"""
    if gradio_padim_model is None:
        return "모델이 로드되지 않았습니다. 학습을 먼저 실행하거나 모델 경로를 확인하세요.", None, None

    # PIL 이미지를 PyTorch 텐서로 변환 및 전처리
    img_tensor = gradio_test_transform(input_image_pil).unsqueeze(0).to(device) # 배치 차원 추가

    # PaDiM 모델로 추론
    image_score, anomaly_map_array = gradio_padim_model.predict(img_tensor)

    # 결과 처리
    score = image_score[0] # 단일 이미지이므로 첫 번째 점수 사용
    anomaly_map_single = anomaly_map_array[0] # 단일 이미지의 이상 맵

    # 이상 맵 시각화 (Heatmap)
    # 0-1 범위로 정규화 후 컬러맵 적용
    anomaly_map_normalized = (anomaly_map_single - np.min(anomaly_map_single)) / \
                             (np.max(anomaly_map_single) - np.min(anomaly_map_single) + 1e-6)

    heatmap_img = cv2.applyColorMap(np.uint8(255 * anomaly_map_normalized), cv2.COLORMAP_JET)
    heatmap_img = cv2.cvtColor(heatmap_img, cv2.COLOR_BGR2RGB) # Matplotlib/Gradio용 RGB

    # 원본 이미지에 히트맵 오버레이 (PIL 이미지를 NumPy로 변환 후 사용)
    original_np = np.array(input_image_pil.resize((IMG_SIZE, IMG_SIZE)))
    if original_np.shape[2] == 4: # RGBA -> RGB
        original_np = original_np[..., :3]

    heatmap_resized = cv2.resize(heatmap_img, (original_np.shape[1], original_np.shape[0]))
    overlayed_image = cv2.addWeighted(original_np, 0.6, heatmap_resized, 0.4, 0)

    result_text = f"이상 점수(Anomaly Score): {score:.4f}"
    if score > 15: # 임계값 예시 (데이터와 모델에 따라 크게 달라짐, 튜닝 필요!)
        result_text += "\n판정: 비정상 (Anomaly)"
    else:
        result_text += "\n판정: 정상 (Normal)"

    return result_text, heatmap_img, overlayed_image



In [ ]:
import gradio as gr

In [ ]:
# --- Gradio 인터페이스 생성 및 실행 ---
if gradio_padim_model is not None: # 모델이 성공적으로 로드된 경우에만 인터페이스 실행
    iface = gr.Interface(
        fn=padim_inference_gradio,
        inputs=gr.Image(type="pil", label="검사할 이미지 업로드"),
        outputs=[
            gr.Textbox(label="결과"),
            gr.Image(type="numpy", label="이상 맵 (Anomaly Map Heatmap)"),
            gr.Image(type="numpy", label="오버레이된 이미지 (Overlayed Image)")
        ],
        title="PaDiM 이상 탐지 시스템 (VisA Dataset)",
        description="이미지를 업로드하여 정상/비정상 여부를 판별하고 이상 영역을 확인합니다. (백본: WideResNet50)",
        examples=[ # 예제 이미지가 있다면 경로를 지정 (Gradio에서 클릭 가능)
        ] if os.path.exists(os.path.join(DATA_PATH, CATEGORY_NAME, "test")) else None # 예제 폴더가 있을 때만 추가
    )

    iface.launch(share=True) # share=True로 하면 외부에서 접속 가능한 링크 생성 (주의해서 사용)
else:
    print("-" * 50)
    print("!!! PaDiM 모델이 로드되지 않았습니다. Gradio 인터페이스를 시작할 수 없습니다. !!!")
    print("!!! 셀 7에서 모델 학습 및 저장이 성공적으로 완료되었는지, SAVED_MODEL_PATH가 올바른지 확인해주세요. !!!")
    print("-" * 50)


# 추가 비교 실험

## 1. 실행 설정

`COMPARE_DATA_PATH`를 VisA 경로로 지정한다. 같은 데이터·시드·학습 조건을 사용하며, 결과는 `comparison_outputs/`에 저장한다.

In [ ]:
from pathlib import Path
import gc
import hashlib
import importlib.metadata
import json
import math
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve

# 원본의 CSV 기준 정상 80% / 나머지 정상 + 모든 이상 분할을 그대로 사용합니다.
# 본 비교에서는 1번(AE), 2번(PaDiM) 외의 모델/데이터 개선을 추가하지 않습니다.
COMPARE_DATA_PATH = Path('./VisA')
COMPARE_CATEGORY = 'pcb4'
COMPARE_OUTPUT_DIR = Path('./comparison_outputs')
COMPARE_IMAGE_SIZE = 256
COMPARE_BATCH_SIZE = 16  # 원본 VisAAnomalyDetectionSystem의 비교 실행과 동일
COMPARE_AE_EPOCHS = 50
COMPARE_LATENT_DIM = 128
COMPARE_TRAIN_RATIO = 0.8
COMPARE_SEED = 42
COMPARE_NUM_WORKERS = 0  # Windows 노트북에서 재현하기 쉬운 기본값

RUN_AE_COMPARISON = True
RUN_PADIM_COMPARISON = True

COMPARE_MEAN = [0.485, 0.456, 0.406]
COMPARE_STD = [0.229, 0.224, 0.225]
COMPARE_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
COMPARE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def seed_comparison(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_comparison(COMPARE_SEED)
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.18,
    'font.size': 10,
})
print('Device:', COMPARE_DEVICE)
print('Results:', COMPARE_OUTPUT_DIR.resolve())

## 2. 데이터와 전처리

정상 이미지 80%로 학습하고, 나머지 정상과 모든 이상 이미지로 시험한다.
기존 AE·PaDiM은 ImageNet 정규화를 사용하고, 개선 AE는 입력을 0~1로 유지한다.

In [ ]:
def build_comparison_records(root, category, train_ratio=0.8):
    """원본 CSV 경로와 행 순서 분할을 유지하되 누락 파일은 명시적으로 알립니다."""
    root = Path(root).expanduser().resolve()
    csv_path = root / category / 'image_anno.csv'
    if not csv_path.exists():
        raise FileNotFoundError(
            f'{csv_path}가 없습니다. COMPARE_DATA_PATH를 기존 VisA 경로로 수정하세요. '
            '원본과 같은 비교를 위해 이 섹션은 image_anno.csv를 사용합니다.'
        )
    if not 0 < train_ratio < 1:
        raise ValueError('train_ratio must be between 0 and 1.')
    table = pd.read_csv(csv_path)
    required = {'image', 'label'}
    if not required.issubset(table.columns):
        raise ValueError(f'CSV에 필요한 컬럼이 없습니다: {required - set(table.columns)}')

    normal = table[table['label'] == 'normal']
    anomaly = table[table['label'] != 'normal']
    train_size = int(len(normal) * train_ratio)
    parts = {
        'train': normal.iloc[:train_size],
        'test': pd.concat([normal.iloc[train_size:], anomaly], ignore_index=True),
    }
    records = {}
    missing = []
    for split, frame in parts.items():
        rows = []
        for row in frame.to_dict('records'):
            relative = str(row['image']).replace('\\', '/')
            image_path = (root / relative if relative.startswith(category + '/')
                          else root / category / relative).resolve()
            if not image_path.is_file():
                missing.append(str(image_path))
                continue
            rows.append({
                'path': str(image_path),
                'label': int(row['label'] != 'normal'),
                'defect_type': str(row['label']),
                'split': split,
            })
        records[split] = rows
    if missing:
        raise FileNotFoundError(f'CSV에 기록된 이미지 {len(missing)}개가 없습니다. 예: {missing[:3]}')
    if len(records['train']) < 2:
        raise ValueError('정상 학습 이미지가 최소 2개 필요합니다.')
    if {r['label'] for r in records['test']} != {0, 1}:
        raise ValueError('AUROC 비교에는 정상과 이상이 모두 있는 시험 세트가 필요합니다.')
    train_paths = {r['path'] for r in records['train']}
    test_paths = {r['path'] for r in records['test']}
    if train_paths & test_paths:
        raise ValueError('학습·시험 경로 중복이 있습니다. 비교를 진행하지 않습니다.')
    return records


class ComparisonImageDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        with Image.open(row['path']) as image:
            image = self.transform(image.convert('RGB'))
        return {'image': image, 'label': row['label'], 'path': row['path']}


def make_comparison_transform(normalize, train):
    steps = [transforms.Resize((COMPARE_IMAGE_SIZE, COMPARE_IMAGE_SIZE))]
    # 원본의 좌우 반전을 두 AE와 모든 PaDiM 설정에서 동일하게 유지합니다.
    if train:
        steps.append(transforms.RandomHorizontalFlip(p=0.5))
    steps.append(transforms.ToTensor())
    if normalize:
        steps.append(transforms.Normalize(COMPARE_MEAN, COMPARE_STD))
    return transforms.Compose(steps)


def make_comparison_loader(records, normalize, train, seed):
    generator = torch.Generator().manual_seed(seed)
    dataset = ComparisonImageDataset(records, make_comparison_transform(normalize, train))
    return DataLoader(
        dataset, batch_size=COMPARE_BATCH_SIZE, shuffle=train,
        num_workers=COMPARE_NUM_WORKERS, generator=generator,
        pin_memory=(COMPARE_DEVICE.type == 'cuda'), drop_last=False,
    )

## 3. 이상 점수와 지표

- 기존 AE: 전체 평균 MSE.
- 개선 AE: 평균·상위 1%·상위 5% 공간 오차.
- 기존 PaDiM: 패치 점수의 최댓값.

AUROC·AUPRC로 분리 성능을 비교한다. `Oracle_Best_F1`은 시험 세트에서 최적 임계값을 고른 참고값으로, 운영 F1과 구분한다.

In [ ]:
def aggregate_comparison_scores(maps, mode='max', fraction=0.01):
    """이미지별 맵에서 평균/최댓값/상위 비율 평균을 계산합니다."""
    flat = np.asarray(maps, dtype=np.float64).reshape(len(maps), -1)
    if not np.isfinite(flat).all():
        raise ValueError('이상 맵에 NaN/Inf가 있습니다.')
    if mode == 'mean':
        return flat.mean(axis=1)
    if mode == 'max':
        return flat.max(axis=1)
    if mode != 'topk' or not 0 < fraction <= 1:
        raise ValueError('mode must be mean/max/topk; fraction must be in (0, 1].')
    k = max(1, math.ceil(flat.shape[1] * fraction))
    return np.partition(flat, flat.shape[1] - k, axis=1)[:, -k:].mean(axis=1)


def comparison_metrics(labels, scores):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if len(labels) != len(scores) or not np.isfinite(scores).all():
        raise ValueError('라벨과 점수의 길이 또는 유한성을 확인하세요.')
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best = int(np.argmax(f1))
    return {
        'AUROC': float(roc_auc_score(labels, scores)),
        'AUPRC': float(auc(recall, precision)),
        'Oracle_Best_F1': float(f1[best]),
        'Oracle_Threshold': float(thresholds[best]),
    }

## 4. 학습·시험 데이터 확인

In [ ]:
comparison_records = build_comparison_records(
    COMPARE_DATA_PATH, COMPARE_CATEGORY, COMPARE_TRAIN_RATIO,
)
comparison_labels = np.array([r['label'] for r in comparison_records['test']], dtype=int)
comparison_split_table = pd.DataFrame(comparison_records['train'] + comparison_records['test'])
comparison_split_table.to_csv(COMPARE_OUTPUT_DIR / 'comparison_split.csv', index=False, encoding='utf-8-sig')
display(pd.crosstab(comparison_split_table['split'], comparison_split_table['label']).rename(
    columns={0: 'normal', 1: 'anomaly'},
))

comparison_rows = []
comparison_predictions = {}
ae_comparison_models = {}
ae_comparison_losses = {}


def record_comparison_result(name, family, scores, config):
    values = comparison_metrics(comparison_labels, scores)
    comparison_rows.append({'method': name, 'family': family, **values})
    comparison_predictions[name] = {'scores': np.asarray(scores), 'config': config}
    print(name, {key: round(value, 4) for key, value in values.items() if key != 'Oracle_Threshold'})

## 5. AE 구조

구조와 Sigmoid 출력은 유지한다. 기존 AE의 정규화 입력과 0~1 출력의 불일치를 없애기 위해, 개선 AE는 입력도 0~1로 맞춘다.

In [ ]:
class ComparisonConvAutoEncoder(nn.Module):
    def __init__(self, input_channels=3, latent_dim=128):
        super(ComparisonConvAutoEncoder, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(input_channels, 32, 4, stride=2, padding=1),  # 128x128
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # 64x64
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),  # 32x32
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),  # 16x16
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, latent_dim, 4, stride=2, padding=1),  # 8x8
            nn.BatchNorm2d(latent_dim),
            nn.ReLU()
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, 4, stride=2, padding=1),  # 16x16
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # 32x32
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # 64x64
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),  # 128x128
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, input_channels, 4, stride=2, padding=1),  # 256x256
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

## 6. AE 학습

기존 AE와 개선 AE를 같은 조건으로 각각 한 번 학습한다. 개선 AE의 세 점수는 같은 모델에서 계산한다.
입력 단위가 다르므로 두 모델의 학습 MSE 크기 자체는 직접 비교하지 않는다.

In [ ]:
def train_comparison_ae(normalize_input, seed):
    # 초기 가중치, 배치 순서, 좌우 반전을 두 실험에서 동일하게 맞춥니다.
    seed_comparison(seed)
    model = ComparisonConvAutoEncoder(input_channels=3, latent_dim=COMPARE_LATENT_DIM).to(COMPARE_DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loader = make_comparison_loader(comparison_records['train'], normalize_input, True, seed)
    losses = []
    model.train()
    for epoch in range(COMPARE_AE_EPOCHS):
        total_loss, total_count = 0.0, 0
        for batch in loader:
            images = batch['image'].to(COMPARE_DEVICE)
            optimizer.zero_grad(set_to_none=True)
            reconstructed = model(images)
            loss = F.mse_loss(reconstructed, images)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach()) * len(images)
            total_count += len(images)
        losses.append(total_loss / total_count)
        if epoch == 0 or (epoch + 1) % 10 == 0 or epoch + 1 == COMPARE_AE_EPOCHS:
            print(f'normalize={normalize_input}, epoch={epoch + 1}/{COMPARE_AE_EPOCHS}, loss={losses[-1]:.6f}')
    return model.eval(), losses


def predict_comparison_ae(model, normalize_input):
    loader = make_comparison_loader(comparison_records['test'], normalize_input, False, COMPARE_SEED)
    collected = {name: [] for name in ['mean', 'top1pct', 'top5pct']}
    seen_labels = []
    with torch.inference_mode():
        for batch in loader:
            images = batch['image'].to(COMPARE_DEVICE)
            reconstructed = model(images)
            # 채널 평균 후 위치별 오차를 계산: 상위 영역을 실제 공간 위치 단위로 선택
            error_map = F.mse_loss(reconstructed, images, reduction='none').mean(dim=1).cpu().numpy()
            collected['mean'].append(aggregate_comparison_scores(error_map, 'mean'))
            collected['top1pct'].append(aggregate_comparison_scores(error_map, 'topk', 0.01))
            collected['top5pct'].append(aggregate_comparison_scores(error_map, 'topk', 0.05))
            seen_labels.extend(batch['label'].tolist())
    assert np.array_equal(seen_labels, comparison_labels), 'AE 시험 순서가 바뀌었습니다.'
    return {name: np.concatenate(values) for name, values in collected.items()}


if RUN_AE_COMPARISON:
    for version, normalized in [('original', True), ('range_fixed', False)]:
        model, losses = train_comparison_ae(normalized, COMPARE_SEED)
        scores = predict_comparison_ae(model, normalized)
        ae_comparison_models[version] = model.cpu()
        ae_comparison_losses[version] = losses
        for aggregation, values in scores.items():
            if version == 'original' and aggregation != 'mean':
                continue  # 기존 AE 기준은 원본의 전체 평균 점수만 사용
            record_comparison_result(
                f'AE_{version}_{aggregation}', 'AE', values,
                {'normalized_input': normalized, 'aggregation': aggregation,
                 'epochs': COMPARE_AE_EPOCHS, 'seed': COMPARE_SEED},
            )
        if COMPARE_DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
else:
    print('AE comparison skipped by configuration.')

## 7. 복원 이미지 비교

두 출력을 RGB로 표시하고, 표시용 오차도 같은 RGB 공간에서 계산한다. 오차 색 범위는 같은 행에서 동일하다.

In [ ]:
def plot_ae_range_comparison(sample_indices=None):
    if not {'original', 'range_fixed'}.issubset(ae_comparison_models):
        print('두 AE 실험을 실행한 뒤 복원 결과를 비교할 수 있습니다.')
        return
    if sample_indices is None:
        normal = np.flatnonzero(comparison_labels == 0)[:2]
        anomaly = np.flatnonzero(comparison_labels == 1)[:2]
        sample_indices = np.concatenate([normal, anomaly])
    figure, axes = plt.subplots(len(sample_indices), 5, figsize=(15, 3 * len(sample_indices)), squeeze=False)
    mean = torch.tensor(COMPARE_MEAN).view(3, 1, 1)
    std = torch.tensor(COMPARE_STD).view(3, 1, 1)
    for row_idx, sample_idx in enumerate(sample_indices):
        record = comparison_records['test'][int(sample_idx)]
        with Image.open(record['path']) as image:
            original = make_comparison_transform(False, False)(image.convert('RGB'))
        reconstructions = {}
        for version, normalized in [('original', True), ('range_fixed', False)]:
            model_input = (original - mean) / std if normalized else original
            with torch.inference_mode():
                reconstructed = ae_comparison_models[version](model_input.unsqueeze(0))[0]
            # 기존 AE의 출력은 정규화된 공간의 값이므로 RGB로 되돌려 표시합니다.
            reconstructed_rgb = (reconstructed * std + mean if normalized else reconstructed).clamp(0, 1)
            reconstructions[version] = reconstructed_rgb
        errors = {key: ((value - original) ** 2).mean(0).numpy()
                  for key, value in reconstructions.items()}
        shared_max = max(max(float(value.max()) for value in errors.values()), 1e-8)
        images = [original, reconstructions['original'], reconstructions['range_fixed']]
        for column, image in enumerate(images):
            axes[row_idx, column].imshow(image.permute(1, 2, 0).numpy())
        for column, key in [(3, 'original'), (4, 'range_fixed')]:
            axes[row_idx, column].imshow(errors[key], cmap='magma', vmin=0, vmax=shared_max)
        for axis in axes[row_idx]:
            axis.set_axis_off()
        axes[row_idx, 0].set_title(f"Input: {'Anomaly' if record['label'] else 'Normal'}")
        for column, title in enumerate(['Original AE: RGB', 'Range-fixed AE: RGB', 'Original RGB error', 'Fixed RGB error'], start=1):
            axes[row_idx, column].set_title(title)
    figure.suptitle('AE reconstructions and RGB-space errors (same error scale within each row)', y=1.01)
    figure.tight_layout()
    figure.savefig(COMPARE_OUTPUT_DIR / 'ae_reconstruction_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_ae_range_comparison()

## 8. 기존 PaDiM

WideResNet50-2 V1의 layer2·3 → 28×28 특징 맵 → 무작위 투영 100차원 → 위치별 정상 분포 → 마할라노비스 거리 최댓값.
공분산 대각 정규화는 1e-4로 유지한다.

In [ ]:
class OriginalPaDiMBackbone(nn.Module):
    """기존 PaDiM의 WideResNet50-2 V1 / layer2·3 특징만 반환합니다."""
    def __init__(self):
        super().__init__()
        if hasattr(models, 'Wide_ResNet50_2_Weights'):
            network = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1)
        else:
            network = models.wide_resnet50_2(pretrained=True)
        self.stem = nn.Sequential(network.conv1, network.bn1, network.relu, network.maxpool)
        self.layer1, self.layer2, self.layer3 = network.layer1, network.layer2, network.layer3
        for parameter in self.parameters():
            parameter.requires_grad_(False)
        self.eval()

    def forward(self, images):
        layer1 = self.layer1(self.stem(images))
        layer2 = self.layer2(layer1)
        layer3 = self.layer3(layer2)
        return {'layer2': layer2, 'layer3': layer3}


class OriginalPaDiMBaseline:
    """기존 설정 고정: layer2·3, 무작위 투영 100차원, 1e-4, max."""
    def __init__(self, backbone, seed=42):
        self.backbone = backbone
        self.seed = seed
        self.patch_size = 28
        self.dimensions = 100
        self.regularization = 1e-4
        self.projection_matrix = None
        self.mean = None
        self.covariance = None

    def patch_features(self, images):
        features = self.backbone(images)
        parts = []
        for name in ('layer2', 'layer3'):
            resized = F.adaptive_avg_pool2d(features[name], (self.patch_size, self.patch_size))
            parts.append(resized.flatten(2).transpose(1, 2))
        combined = torch.cat(parts, dim=-1)
        if self.projection_matrix is None:
            generator = torch.Generator(device='cpu').manual_seed(self.seed)
            projection = torch.randn(combined.shape[-1], self.dimensions, generator=generator)
            self.projection_matrix = F.normalize(projection, dim=0).to(images.device)
        return combined @ self.projection_matrix

    def fit(self, loader):
        # 기존 비교 코드의 표본 평균·불편 공분산 정의를 그대로 유지합니다.
        count, running_mean, running_m2 = 0, None, None
        self.projection_matrix = None
        with torch.inference_mode():
            for batch in loader:
                features = self.patch_features(batch['image'].to(COMPARE_DEVICE)).float()
                batch_size = len(features)
                batch_mean = features.mean(0)
                centered = features - batch_mean
                batch_m2 = torch.einsum('bpd,bpe->pde', centered, centered)
                if count == 0:
                    running_mean, running_m2 = batch_mean, batch_m2
                else:
                    combined_count = count + batch_size
                    delta = batch_mean - running_mean
                    running_m2 = running_m2 + batch_m2 + torch.einsum('pd,pe->pde', delta, delta) * (count * batch_size / combined_count)
                    running_mean = running_mean + delta * (batch_size / combined_count)
                count += batch_size
        if count < 2:
            raise ValueError('공분산 계산에는 최소 2개의 정상 이미지가 필요합니다.')
        self.mean = running_mean
        covariance = running_m2 / (count - 1)
        self.covariance = 0.5 * (covariance + covariance.transpose(-1, -2))
        if not torch.isfinite(self.covariance).all():
            raise ValueError('공분산에 NaN/Inf가 있습니다.')
        return self

    def predict(self, loader):
        identity = torch.eye(self.dimensions, device=self.mean.device, dtype=self.mean.dtype)
        covariance = self.covariance + self.regularization * identity
        try:
            inverse = torch.linalg.inv(covariance)
        except RuntimeError:
            inverse = torch.linalg.pinv(covariance)
        scores, seen_labels = [], []
        with torch.inference_mode():
            for batch in loader:
                difference = self.patch_features(batch['image'].to(COMPARE_DEVICE)).float() - self.mean
                squared = torch.einsum('bpd,pde,bpe->bp', difference, inverse, difference)
                distances = squared.clamp_min(0).sqrt()
                scores.append(distances.max(dim=1).values.cpu().numpy())
                seen_labels.extend(batch['label'].tolist())
        assert np.array_equal(seen_labels, comparison_labels), 'PaDiM 시험 순서가 바뀌었습니다.'
        return np.concatenate(scores)

## 9. 기존 PaDiM 실행

같은 학습·시험 데이터로 기존 설정 하나만 실행한다.

In [ ]:
if RUN_PADIM_COMPARISON:
    seed_comparison(COMPARE_SEED)
    original_padim_backbone = OriginalPaDiMBackbone().to(COMPARE_DEVICE).eval()
    # 백본 생성 시 난수 소비가 학습 증강에 영향을 주지 않도록 다시 고정합니다.
    seed_comparison(COMPARE_SEED)
    padim_train_loader = make_comparison_loader(comparison_records['train'], True, True, COMPARE_SEED)
    padim_test_loader = make_comparison_loader(comparison_records['test'], True, False, COMPARE_SEED)
    original_padim = OriginalPaDiMBaseline(original_padim_backbone, COMPARE_SEED).fit(padim_train_loader)
    original_padim_scores = original_padim.predict(padim_test_loader)
    record_comparison_result(
        'PaDiM_original', 'PaDiM', original_padim_scores,
        {'backbone': 'wide_resnet50_2_IMAGENET1K_V1', 'layers': ['layer2', 'layer3'],
         'reduction': 'projection', 'dimensions': 100, 'regularization': 1e-4,
         'aggregation': 'max', 'patch_size': 28, 'seed': COMPARE_SEED},
    )
    original_padim_backbone.cpu()
    del original_padim
    gc.collect()
    if COMPARE_DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
else:
    print('Original PaDiM baseline skipped by configuration.')

## 10. 비교 결과

| 비교 | Δ 계산 기준 |
|---|---|
| 기존 AE vs 개선 AE | `AE_original_mean` |
| 개선 AE vs 기존 PaDiM | `PaDiM_original` |

Δ가 양수면 기준보다 높고, 음수면 낮다.

In [ ]:
def get_ae_comparison_specs():
    improved_names = ['AE_range_fixed_mean', 'AE_range_fixed_top1pct', 'AE_range_fixed_top5pct']
    return [
        {'key': 'original_ae_vs_improved_ae',
         'title': 'Original AE vs improved AE',
         'baseline': 'AE_original_mean',
         'methods': ['AE_original_mean', *improved_names]},
        {'key': 'improved_ae_vs_original_padim',
         'title': 'Improved AE vs original PaDiM',
         'baseline': 'PaDiM_original',
         'methods': ['PaDiM_original', *improved_names]},
    ]


def make_ae_comparison_table(results, spec):
    if results.empty:
        return pd.DataFrame()
    indexed = results.set_index('method')
    if spec['baseline'] not in indexed.index:
        return pd.DataFrame()
    names = [name for name in spec['methods'] if name in indexed.index]
    selected = indexed.loc[names].copy()
    baseline = indexed.loc[spec['baseline']]
    for metric in ['AUROC', 'AUPRC', 'Oracle_Best_F1']:
        selected['delta_' + metric] = selected[metric] - baseline[metric]
    selected['baseline'] = spec['baseline']
    return selected.reset_index()


comparison_results = pd.DataFrame(comparison_rows)
comparison_tables = {}
if not comparison_results.empty:
    comparison_results.to_csv(COMPARE_OUTPUT_DIR / 'metrics.csv', index=False, encoding='utf-8-sig')
    for spec in get_ae_comparison_specs():
        table = make_ae_comparison_table(comparison_results, spec)
        comparison_tables[spec['key']] = table
        if table.empty:
            print(spec['title'], ': 기준 모델이 실행되지 않아 비교를 생략합니다.')
            continue
        print('\n' + spec['title'] + ' | delta baseline: ' + spec['baseline'])
        display(table.round(4))
        table.to_csv(COMPARE_OUTPUT_DIR / (spec['key'] + '.csv'), index=False, encoding='utf-8-sig')

    prediction_rows = []
    for name, result in comparison_predictions.items():
        for record, score in zip(comparison_records['test'], result['scores']):
            prediction_rows.append({'method': name, 'path': record['path'],
                                    'label': record['label'], 'score': float(score)})
    pd.DataFrame(prediction_rows).to_csv(COMPARE_OUTPUT_DIR / 'predictions.csv', index=False, encoding='utf-8-sig')
    packages = {}
    for package in ['torch', 'torchvision', 'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'Pillow']:
        try:
            packages[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            packages[package] = 'unavailable'
    split_fingerprint = hashlib.sha256(
        '\n'.join(f"{row['split']}|{row['path']}|{row['label']}" for row in comparison_split_table.to_dict('records')).encode('utf-8')
    ).hexdigest()
    manifest = {
        'scope': 'AE improvements only; original PaDiM is a fixed reference',
        'category': COMPARE_CATEGORY, 'data_path': str(COMPARE_DATA_PATH.resolve()),
        'seed': COMPARE_SEED, 'image_size': COMPARE_IMAGE_SIZE,
        'train_ratio': COMPARE_TRAIN_RATIO, 'ae_epochs': COMPARE_AE_EPOCHS,
        'device': str(COMPARE_DEVICE), 'packages': packages,
        'split_sha256': split_fingerprint,
        'comparisons': get_ae_comparison_specs(),
        'configs': {name: result['config'] for name, result in comparison_predictions.items()},
        'metric_note': 'Oracle_Best_F1 is an exploratory test-set optimum, not held-out operating F1.',
    }
    (COMPARE_OUTPUT_DIR / 'experiment_manifest.json').write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8',
    )
else:
    print('No experiments were run.')

## 11. 두 비교의 성능·ROC 그래프

In [ ]:
def plot_ae_comparison_metrics(tables):
    for spec in get_ae_comparison_specs():
        table = tables.get(spec['key'], pd.DataFrame())
        if table.empty:
            continue
        names = table['method'].tolist()
        positions = np.arange(len(table))
        colors = ['#64748b' if name == spec['baseline'] else '#2563a6' for name in names]
        figure, axes = plt.subplots(1, 3, figsize=(16, 4.1), sharey=True)
        for axis, metric in zip(axes, ['AUROC', 'AUPRC', 'Oracle_Best_F1']):
            values = table[metric].to_numpy()
            axis.barh(positions, values, color=colors)
            axis.set_xlim(0, 1.09)
            axis.set_xticks([0, 0.25, 0.5, 0.75, 1])
            axis.set_yticks(positions, labels=names)
            axis.set_title(metric)
            axis.set_xlabel('Score (0-1)')
            for position, value in zip(positions, values):
                axis.text(value + 0.012, position, f'{value:.3f}', va='center', fontsize=9)
        axes[0].invert_yaxis()
        figure.suptitle(spec['title'] + ' | ' + COMPARE_CATEGORY, y=1.01)
        figure.tight_layout()
        figure.savefig(COMPARE_OUTPUT_DIR / (spec['key'] + '_metrics.png'), dpi=150, bbox_inches='tight')
        plt.show()


def plot_ae_comparison_roc():
    colors = ['#64748b', '#2563a6', '#b66a18', '#7c3e96']
    styles = ['--', '-', '-.', ':']
    for spec in get_ae_comparison_specs():
        names = [name for name in spec['methods'] if name in comparison_predictions]
        if spec['baseline'] not in names or len(names) < 2:
            continue
        figure, axis = plt.subplots(figsize=(8, 6))
        for name, color, style in zip(names, colors, styles):
            scores = comparison_predictions[name]['scores']
            fpr, tpr, _ = roc_curve(comparison_labels, scores)
            axis.plot(fpr, tpr, color=color, linestyle=style,
                      label=f'{name} (AUC={roc_auc_score(comparison_labels, scores):.3f})')
        axis.plot([0, 1], [0, 1], color='#9ca3af', linestyle=':', label='Random ranking')
        axis.set(xlim=(0, 1), ylim=(0, 1.02), xlabel='False positive rate',
                 ylabel='True positive rate', title=spec['title'])
        axis.legend(loc='lower right', fontsize=8)
        figure.tight_layout()
        figure.savefig(COMPARE_OUTPUT_DIR / (spec['key'] + '_roc.png'), dpi=150, bbox_inches='tight')
        plt.show()


plot_ae_comparison_metrics(comparison_tables)
plot_ae_comparison_roc()